# 02 — FOV46 extraction and interactive QC

FOV46 is extracted from the **raw** AnnData/SpatialData. First run extraction, then run the current QC script. The final cells make it easy to inspect and override thresholds before moving to Leiden.

In [1]:
from pathlib import Path
import sys, os

# Notebook lives in PROJECT_ROOT/jupyter/.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "jupyter" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
assert (PROJECT_ROOT / "scripts").exists(), "Run this notebook from PROJECT_ROOT/jupyter or PROJECT_ROOT."

PROJECT_ROOT: /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised
Python: /home/jqu/.conda/envs/spatialdata/bin/python


In [ ]:
%run ../scripts/04_extract_fov46.py

In [ ]:
%run ../scripts/04a_qc_fov46.py

## Inspect the thresholds actually used

In [2]:
import json, pandas as pd, scanpy as sc
from IPython.display import display
TAB=PROJECT_ROOT/"results/FOV46/tables"
thr=json.loads((TAB/"recommended_qc_thresholds.json").read_text())
print(json.dumps(thr,indent=2))
display(pd.read_csv(TAB/"qc_threshold_sweep.csv"))

{
  "min_counts": 75,
  "min_genes": 32,
  "max_counts": 2400,
  "max_control_fraction": 0.007182337045669567,
  "min_cell_area": 3337.0824715846375,
  "max_cell_area": 14913.890936282127
}


,min_counts,min_genes,max_control_fraction,n_cells,retained_fraction,min_fov_retention,median_fov_retention,fov_retention_iqr
0,10,5,NaN,1099,0.994570,0.994570,0.994570,0.0
1,10,5,0.05,1099,0.994570,0.994570,0.994570,0.0
2,10,5,0.10,1099,0.994570,0.994570,0.994570,0.0
3,10,5,0.20,1099,0.994570,0.994570,0.994570,0.0
4,10,10,NaN,1097,0.992760,0.992760,0.992760,0.0
...,...,...,...,...,...,...,...,...
115,100,20,0.20,980,0.886878,0.886878,0.886878,0.0
116,100,30,NaN,960,0.868778,0.868778,0.868778,0.0
117,100,30,0.05,960,0.868778,0.868778,0.868778,0.0
118,100,30,0.10,960,0.868778,0.868778,0.868778,0.0


## Inspect QC retention and distributions

In [3]:
rawq=sc.read_h5ad(PROJECT_ROOT/"results/FOV46/GSM9046088_FOV46_raw_with_qc_flags.h5ad")
print(rawq.obs["qc_pass"].value_counts(dropna=False))
print("Retention:", rawq.obs["qc_pass"].mean())
cols=[c for c in ["total_counts","n_genes_by_counts","control_fraction","Area"] if c in rawq.obs]
display(rawq.obs.groupby("qc_pass")[cols].describe().T)

qc_pass
True     938
False    167
Name: count, dtype: int64
Retention: 0.848868778280543


qc_pass                         False         True 
total_counts      count    167.000000    938.000000
                  mean     147.203598    419.191895
                  std      204.724457    233.089539
                  min        3.000000     77.000000
                  25%       53.000000    247.000000
                  50%       88.000000    380.000000
                  75%      155.500000    551.500000
                  max     1400.000000   1482.000000
n_genes_by_counts count    167.000000    938.000000
                  mean      47.676647    122.841151
                  std       53.780037     55.821060
                  min        2.000000     32.000000
                  25%       20.000000     81.000000
                  50%       29.000000    116.000000
                  75%       44.500000    154.000000
                  max      321.000000    339.000000
control_fraction  count    167.000000    938.000000
                  mean       0.000807      0.000518
                  std        0.002909      0.001272
                  min        0.000000      0.000000
                  25%        0.000000      0.000000
                  50%        0.000000      0.000000
                  75%        0.000000      0.000000
                  max        0.022222      0.006897
Area              count    167.000000    938.000000
                  mean    6185.395210   7581.226013
                  std     3601.273036   2473.504268
                  min     1788.000000   3362.000000
                  25%     3554.000000   5709.750000
                  50%     5628.000000   7226.000000
                  75%     7394.000000   9090.500000
                  max    19291.000000  14896.000000

## Optional threshold override
Edit only if QC inspection supports a change. This cell creates a new QC-filtered object using explicit thresholds, without modifying the production script.

In [ ]:
# Example — uncomment/edit to test an alternative QC set.
# from src.qc import apply_qc
# test = rawq.copy()
# test_thr = dict(thr)
# test_thr["min_counts"] = 75
# test_thr["min_genes"] = 32
# apply_qc(test, test_thr, use_per_fov_flags=False)
# print(test.obs["qc_pass"].value_counts())